# Multi-Component Flash Separation (BTX)

Simulating an isothermal flash drum for a ternary **benzene–toluene–p-xylene** (BTX) mixture.
The `MultiComponentFlash` block uses Raoult's law with Antoine correlations to compute K-values
and solves the Rachford-Rice equation via Brent's method.

This example is inspired by [MiniSim's SimpleFlash example](https://github.com/Nukleon84/MiniSim/blob/master/doc/SimpleFlash.ipynb),
adapted to PathSim's dynamic simulation framework.

**Feed conditions:**
- 10 mol/s total flow
- 50% benzene, 10% toluene, 40% p-xylene (molar)
- 1 atm pressure
- Temperature sweep: 340 K → 420 K

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pathsim import Simulation, Connection
from pathsim.blocks import Source, Scope

from pathsim_chem.process import MultiComponentFlash

## Flash Drum Setup

The `MultiComponentFlash` block defaults to BTX Antoine parameters (ln form, Pa, K):

| Component | A | B | C |
|-----------|-------|---------|--------|
| Benzene | 20.7936 | 2788.51 | -52.36 |
| Toluene | 20.9064 | 3096.52 | -53.67 |
| p-Xylene | 20.9891 | 3346.65 | -57.84 |

We feed the drum with constant composition and pressure while ramping temperature
to observe the transition from all-liquid through two-phase to all-vapor.

In [ ]:
# Create the flash drum (3 components, BTX defaults)
flash = MultiComponentFlash(N_comp=3, holdup=100.0)

# Feed sources
F_feed = Source(func=lambda t: 10.0)          # 10 mol/s
z_benzene = Source(func=lambda t: 0.5)        # 50% benzene
z_toluene = Source(func=lambda t: 0.1)        # 10% toluene (p-xylene = 40% inferred)
T_sweep = Source(func=lambda t: 340.0 + t)    # ramp 340 -> 420 K
P_feed = Source(func=lambda t: 101325.0)      # 1 atm

# Record all outputs: V_rate, L_rate, y_benzene, y_toluene, x_benzene, x_toluene
scp = Scope(labels=["V_rate", "L_rate", "y_benz", "y_tol", "x_benz", "x_tol"])

sim = Simulation(
    blocks=[F_feed, z_benzene, z_toluene, T_sweep, P_feed, flash, scp],
    connections=[
        Connection(F_feed, flash),            # F -> port 0
        Connection(z_benzene, flash[1]),       # z_1 (benzene) -> port 1
        Connection(z_toluene, flash[2]),       # z_2 (toluene) -> port 2
        Connection(T_sweep, flash[3]),         # T -> port 3
        Connection(P_feed, flash[4]),          # P -> port 4
        Connection(flash, scp),                # V_rate -> scope 0
        Connection(flash[1], scp[1]),          # L_rate -> scope 1
        Connection(flash[2], scp[2]),          # y_benzene -> scope 2
        Connection(flash[3], scp[3]),          # y_toluene -> scope 3
        Connection(flash[4], scp[4]),          # x_benzene -> scope 4
        Connection(flash[5], scp[5]),          # x_toluene -> scope 5
    ],
    dt=0.5,
)

sim.run(80)

## Results: Flow Rates

As temperature increases, the vapor fraction grows. Below the bubble point the drum produces
only liquid; above the dew point it produces only vapor.

In [ ]:
time, signals = scp.read()
T = 340.0 + time  # temperature axis

V_rate, L_rate = signals[0], signals[1]
y_benz, y_tol = signals[2], signals[3]
x_benz, x_tol = signals[4], signals[5]

# Infer p-xylene fractions
y_xyl = 1.0 - y_benz - y_tol
x_xyl = 1.0 - x_benz - x_tol

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Flow rates
ax = axes[0]
ax.plot(T, V_rate, label="Vapor")
ax.plot(T, L_rate, label="Liquid")
ax.set_xlabel("Temperature [K]")
ax.set_ylabel("Flow rate [mol/s]")
ax.set_title("Flash Drum Flow Rates")
ax.legend()
ax.grid(True, alpha=0.3)

# Vapor compositions
ax = axes[1]
ax.plot(T, y_benz, label="Benzene")
ax.plot(T, y_tol, label="Toluene")
ax.plot(T, y_xyl, label="p-Xylene")
ax.set_xlabel("Temperature [K]")
ax.set_ylabel("Vapor mole fraction")
ax.set_title("Vapor Composition")
ax.legend()
ax.grid(True, alpha=0.3)

# Liquid compositions
ax = axes[2]
ax.plot(T, x_benz, label="Benzene")
ax.plot(T, x_tol, label="Toluene")
ax.plot(T, x_xyl, label="p-Xylene")
ax.set_xlabel("Temperature [K]")
ax.set_ylabel("Liquid mole fraction")
ax.set_title("Liquid Composition")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Results: VLE Diagram

Plot the vapor vs liquid composition for each component across the temperature sweep.
The diagonal represents equal vapor and liquid composition — deviation from it shows
the separation achieved by the flash.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, xi, yi, name in zip(axes,
                             [x_benz, x_tol, x_xyl],
                             [y_benz, y_tol, y_xyl],
                             ["Benzene", "Toluene", "p-Xylene"]):
    ax.plot(xi, yi, ".", markersize=3)
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_xlabel(f"$x$ ({name})")
    ax.set_ylabel(f"$y$ ({name})")
    ax.set_title(f"{name} (x, y)-Diagram")
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Fixed-Temperature Flash at 380 K

For a direct comparison with MiniSim's result (which solves at steady state),
we run a fixed-temperature flash and let the holdup reach equilibrium.

In [ ]:
flash2 = MultiComponentFlash(N_comp=3, holdup=100.0)

F_src = Source(func=lambda t: 10.0)
z1_src = Source(func=lambda t: 0.5)
z2_src = Source(func=lambda t: 0.1)
T_src = Source(func=lambda t: 380.0)       # fixed at 380 K (~107 °C)
P_src = Source(func=lambda t: 101325.0)

scp2 = Scope(labels=["V_rate", "L_rate", "y_benz", "y_tol", "x_benz", "x_tol"])

sim2 = Simulation(
    blocks=[F_src, z1_src, z2_src, T_src, P_src, flash2, scp2],
    connections=[
        Connection(F_src, flash2),
        Connection(z1_src, flash2[1]),
        Connection(z2_src, flash2[2]),
        Connection(T_src, flash2[3]),
        Connection(P_src, flash2[4]),
        Connection(flash2, scp2),
        Connection(flash2[1], scp2[1]),
        Connection(flash2[2], scp2[2]),
        Connection(flash2[3], scp2[3]),
        Connection(flash2[4], scp2[4]),
        Connection(flash2[5], scp2[5]),
    ],
    dt=0.5,
)

sim2.run(100)  # let it reach steady state

time2, signals2 = scp2.read()

# Extract final steady-state values
V_ss = signals2[0][-1]
L_ss = signals2[1][-1]
y_benz_ss = signals2[2][-1]
y_tol_ss = signals2[3][-1]
x_benz_ss = signals2[4][-1]
x_tol_ss = signals2[5][-1]

print("BTX Flash at 380 K, 1 atm")
print("=" * 40)
print(f"{'':20s} {'Vapor':>10s} {'Liquid':>10s}")
print(f"{'-'*40}")
print(f"{'Flow rate [mol/s]':20s} {V_ss:10.3f} {L_ss:10.3f}")
print(f"{'Benzene':20s} {y_benz_ss:10.4f} {x_benz_ss:10.4f}")
print(f"{'Toluene':20s} {y_tol_ss:10.4f} {x_tol_ss:10.4f}")
print(f"{'p-Xylene':20s} {1-y_benz_ss-y_tol_ss:10.4f} {1-x_benz_ss-x_tol_ss:10.4f}")

The lighter component (benzene) is enriched in the vapor phase while the heavier
component (p-xylene) concentrates in the liquid — exactly the separation behaviour
expected from VLE. The dynamic formulation reaches the same steady state that
an equation-oriented solver (like MiniSim) finds directly.